# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/api/python/) library.

### Dataset Source
The dataset source is defined by a Croissant schema URL: 

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Dataset ID: {getattr(metadata, '@id', None)}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs with reference to their `@id` identifiers.

In [ ]:
# List all available record sets, fields, and columns by their @id
print("Available Record Sets:")

if hasattr(metadata, 'record_sets'):
    for record_set in metadata.record_sets:
        print(f"- Record set name: {getattr(record_set, 'name', None)}; @id: {getattr(record_set, '@id', None)}")
        if hasattr(record_set, 'fields'):
            print("  Fields:")
            for field in record_set.fields:
                print(f"    - Field name: {getattr(field, 'name', None)}; @id: {getattr(field, '@id', None)}; dataType: {getattr(field, 'data_type', None)}")
                if hasattr(field, 'columns'):
                    for column in field.columns:
                        print(f"      - Column header: {getattr(column, 'header', None)}; @id: {getattr(column, '@id', None)}")
else:
    # For legacy or flat Croissant, check 'record_set'
    record_sets = getattr(metadata, 'record_set', [])
    if not record_sets:
        print("No record sets found in metadata.")
    else:
        for record_set in record_sets:
            print(f"- Record set name: {getattr(record_set, 'name', None)}; @id: {getattr(record_set, '@id', None)}")
            if hasattr(record_set, 'fields'):
                print("  Fields:")
                for field in record_set.fields:
                    print(f"    - Field name: {getattr(field, 'name', None)}; @id: {getattr(field, '@id', None)}; dataType: {getattr(field, 'data_type', None)}")
                    if hasattr(field, 'columns'):
                        for column in field.columns:
                            print(f"      - Column header: {getattr(column, 'header', None)}; @id: {getattr(column, '@id', None)}")

## 3. Data Extraction
Load records from available record sets, referencing each one by its `@id`. The resulting dataframes use field `@id`s as columns for clarity and repeatability.

In [ ]:
# Retrieve all record set IDs from metadata
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    record_sets_list = metadata.record_sets
elif hasattr(metadata, 'record_set'):
    record_sets_list = metadata.record_set
else:
    record_sets_list = []

for record_set in record_sets_list:
    rs_id = getattr(record_set, '@id', None)
    if rs_id:
        record_set_ids.append(rs_id)

if not record_set_ids:
    print("No record sets found in metadata. Please check schema or contact the dataset administrator.")
else:
    print("Record Set @id list:")
    for rid in record_set_ids:
        print(f"- {rid}")

# Load each record set as a dataframe
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded records for Record Set @id: {record_set_id} (Rows: {df.shape[0]}, Columns: {df.shape[1]})")
    except Exception as e:
        print(f"Could not load records for Record Set {record_set_id}: {str(e)}")

# Display available columns for the first record set (by @id)
if record_set_ids:
    first_rs = record_set_ids[0]
    print("Columns in first record set:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate simple processing by filtering, normalizing, and grouping data. All field and record set references use their `@id` for consistency.

In [ ]:
# EDA on the first available record set
import numpy as np
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    
    # Attempt to find a numeric field among columns (based on types or example values)
    numeric_field = None
    # Try via metadata
    matching_rs = None
    for rs in record_sets_list:
        if getattr(rs, '@id', None) == record_set_id:
            matching_rs = rs
            break
    if matching_rs is not None and hasattr(matching_rs, 'fields'):
        for f in matching_rs.fields:
            dtype = getattr(f, 'data_type', '')
            if dtype and any([s in dtype.lower() for s in ['float', 'number', 'integer', 'numeric']]):
                f_id = getattr(f, '@id', None)
                if f_id in df.columns:
                    numeric_field = f_id
                    break
    
    if numeric_field is None:
        # Fallback: guess first column with numeric-looking data
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    
    if numeric_field is not None:
        print(f"Using {numeric_field} as the numeric field for filtering and normalization.")
        threshold = np.nanpercentile(df[numeric_field].values, 80)  # Top 20% as example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        mean_val = filtered_df[numeric_field].mean()
        std_val = filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 1
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean_val) / std_val
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt group-by using first non-numeric field
        group_field = None
        if matching_rs is not None:
            for f in matching_rs.fields:
                dtype = getattr(f, 'data_type', '')
                f_id = getattr(f, '@id', None)
                if f_id != numeric_field and f_id in df.columns and dtype and 'text' in dtype.lower():
                    group_field = f_id
                    break
        if group_field is None:
            # fallback: just pick first object-type or non-numeric column
            for col in df.select_dtypes(include=['object', 'category']).columns:
                if col != numeric_field:
                    group_field = col
                    break
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field} (top 5 groups):")
            display(grouped_df.head())
    else:
        print("No numeric field found for EDA. Please review the schema for available data types.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using their `@id` references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic visualization: Distribution of the selected numeric field
if record_set_ids and numeric_field is not None and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=25)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# If group_field exists, plot boxplot for numeric_field by group_field
if group_field is not None and group_field in df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to explore and process the FAIR^2 rangeland management dataset using the Croissant `mlcroissant` library. Key steps included metadata inspection, dynamic loading of record sets and fields by `@id`, basic EDA, and visualizations referenced by schema. You are encouraged to extend these analyses for your own research and projects based on the dataset's Croissant schema.